# 01. Data preparation: S3 data lake, leakage removal, encoding, chronological split

**Part 1 (local, BP-305):** check the raw data against the contract, run the pipeline, and inspect the files it writes. No AWS access is needed.

**Part 2 (AWS lab, BP-402 and BP-404):** upload the outputs to the S3 data lake. These cells are left for you to write.

Every reusable step lives in `bankpulse_core/` and `features/`; this notebook only calls them.

In [ ]:
# Setup: find the repository root whether the notebook runs from notebooks/ or the repo root.
from pathlib import Path

import pandas as pd

from bankpulse_core.config import load_config
from bankpulse_core.contract import check_rules, check_schema, load_contract, quality_report
from bankpulse_core.io import read_csv, read_json
from bankpulse_core.pipeline import run

ROOT = Path.cwd() if (Path.cwd() / 'config').is_dir() else Path.cwd().parent
config = load_config(ROOT / 'config' / 'data_prep.yaml')
contract = load_contract(config['contract'])
print('raw data:', config['data']['path'])
print('features enabled:', len(config['features']))

## Step 1. Data quality report

The raw file must match `config/contract.yaml`: all 21 columns with the right types and allowed values, and the 4 business rules. Any problem stops here, with every issue listed.

In [ ]:
raw = read_csv(config['data']['path'], sep=config['data']['separator'])
check_schema(raw, contract)
check_rules(raw, contract)

report = quality_report(raw, contract)
summary = pd.Series({k: report[k] for k in ('rows', 'columns', 'duplicate_rows', 'positive_rate')}, dtype=object)
display(summary.to_frame('value'))
display(pd.Series(report['unknown_share'], name='share of "unknown"').to_frame())
print('pdays = 999 (never contacted before):', report['sentinel_share']['pdays'])

## Step 2. Run the pipeline

`pipeline.run()` does the following:
- drops `duration` (it leaks the outcome)
- splits 70 / 15 / 15 in time order
- fits the 12 features on the training split only
- writes the XGBoost files, `feature_schema.json` and `run_metadata.json`

In [ ]:
result = run(config)

split_summary = pd.DataFrame(result['split_summary']).T
split_summary['positive_rate'] = (split_summary['positive_rate'] * 100).round(2).astype(str) + '%'
display(split_summary)
print(result['n_columns'], 'feature columns')

In [ ]:
# Subscription rate per split: the newest customers subscribe far more often (drift).
rates = pd.Series({name: values['positive_rate'] for name, values in result['split_summary'].items()})
ax = (rates * 100).plot.bar(title='Subscription rate by time-ordered split', ylabel='% subscribed', rot=0,
                            color=['#2F6DB5', '#E07B00', '#2E8B57'])
ax.bar_label(ax.containers[0], fmt='%.1f%%');

## Step 3. Inspect the outputs

The files use the built-in XGBoost CSV format: no header, and the label in the first column for train and validation. Each training channel gets its own folder, because SageMaker reads every file under a channel's S3 prefix.

In [ ]:
outputs = pd.DataFrame(
    [(name, Path(path).relative_to(Path(config['output']['processed_dir'])).as_posix(),
      round(Path(path).stat().st_size / 1024, 1)) for name, path in result['outputs'].items()],
    columns=['output', 'file', 'size_kb'])
display(outputs)

In [ ]:
# The first train rows with the schema's column names (the file itself has no header).
schema = read_json(result['outputs']['feature_schema'])
train = read_csv(result['outputs']['train'], header=None)
train.columns = ['label'] + schema['columns']
display(train.iloc[:, :12].head())        # first 12 of the 74 columns; train.head() shows all
print('label values:', sorted(train['label'].unique().tolist()), '| feature columns:', len(schema['columns']))

In [ ]:
# Lineage: which data, code and settings produced these files.
metadata = read_json(result['run_metadata'])
display(pd.Series({
    'run_id': metadata['run_id'],
    'git_commit': (metadata['code']['git_commit'] or '')[:12],
    'uncommitted_changes': metadata['code']['uncommitted_changes'],
    'raw_data_sha256': metadata['inputs']['raw_data']['sha256'][:16] + '...',
    'train_sha256': metadata['outputs']['train']['sha256'][:16] + '...',
}).to_frame('value'))

---
# Part 2. S3 data lake (AWS lab, your SageMaker code)

Before running: start the lab, refresh `~/.aws/credentials`, and run `python scripts\check_aws.py` (see issue BP-400). The kernel must be started from a terminal or VS Code session where `AWS_PROFILE` and `BANKPULSE_ROLE_ARN` are set.

- Bucket: `sagemaker.Session().default_bucket()`
- Prefix: `config['s3']['prefix']` (`bank-campaign-mlops`)
- Helpers you can use: `bankpulse_core.io.upload_file`, `list_objects`

## BP-402. SageMaker session, role and bucket

Create the SageMaker session, read the role from `BANKPULSE_ROLE_ARN` (outside Studio) and get the default bucket.

Screenshot: this cell's output showing the bucket and role.

In [ ]:
# BP-402: your code here

## BP-404. Upload raw data and pipeline outputs to S3

Upload `raw/` plus every file in `result['outputs']` and `run_metadata.json`, keeping the folder layout: `train/`, `validation/`, `batch-input/`, `labels/` and `feature-schema/`, all under `bank-campaign-mlops/`. Then list what is in S3.

Screenshots: the S3 console folder view, and the details of `train/train.csv`.

In [ ]:
# BP-404: your code here